# Notebook 05 — LLM Explanation Generator

## Overview
This notebook demonstrates **Technique 4: Prompt Engineering with a Large Language Model (LLM)**.

We use **Google's Flan-T5-Base** — a seq2seq transformer model — to generate natural-language explanations for why each recommended movie matches the user's mood. This is the 'explainability' component of our system.

## Why LLM Explanations?
Traditional recommendation systems return a ranked list with no explanation. Research shows users trust and engage more with recommendations when they understand the reasoning (Tintarev & Masthoff, 2007). Our system addresses this with natural-language explanations.

## Model: `google/flan-t5-base`
- **Architecture:** T5 (Text-to-Text Transfer Transformer) — encoder-decoder
- **Fine-tuning:** Flan (Fine-tuned Language Net) — instruction-tuned on 1,800+ NLP tasks
- **Parameters:** ~250M (base size, runs on CPU)
- **Available from:** `huggingface.co/google/flan-t5-base`

## Prompt Engineering Techniques Used
1. **Chain-of-thought prompting** — the prompt instructs the model to think step by step before answering
2. **In-context instruction** — the prompt explicitly states the role (movie explainer) and format (2 sentences)
3. **Structured prompt design** — consistent template ensures reproducible output quality

In [1]:
import sys; sys.path.insert(0, '..')
from src.recommender import load_data, recommend
from src.explainer import generate_explanation, _build_prompt
from src.utils import bleu_score
import pandas as pd

print('Libraries loaded.')

Libraries loaded.


## 1. Inspect the Prompt Template

The prompt is the key to quality explanations. We use a chain-of-thought structure:
- **Step 1:** Model thinks about what the user wants
- **Step 2:** Model thinks about what the movie offers
- **Step 3:** Model synthesises both into a 2-sentence explanation

This structured reasoning produces more coherent and relevant explanations than a simple "explain this movie" prompt.

In [2]:
sample_prompt = _build_prompt(
    user_mood='a thrilling sci-fi adventure with great storytelling',
    movie_title='Interstellar',
    movie_genres='Adventure|Drama|Sci-Fi'
)
print('=== PROMPT SENT TO FLAN-T5 ===')
print(sample_prompt)
print('=' * 40)

=== PROMPT SENT TO FLAN-T5 ===
A user is looking for: "a thrilling sci-fi adventure with great storytelling"

Recommended movie: Interstellar
Genres: Adventure|Drama|Sci-Fi

Step 1: Think about what the user wants.
Step 2: Think about what this movie offers.
Step 3: Explain in 2 sentences why this movie matches what the user wants.

Explanation:


## 2. Generate a Single Explanation

In [4]:
explanation = generate_explanation(
    user_mood='a thrilling sci-fi adventure with great storytelling',
    movie_title='Interstellar',
    movie_genres='Adventure|Drama|Sci-Fi'
)
print('Generated explanation:')
print(explanation)

Generated explanation:
Interstellar is a thrilling sci-fi adventure with great storytelling


## 3. Full Pipeline — Recommend + Explain

Here we run the complete end-to-end pipeline: user types a mood → top 3 movies are recommended → each movie gets an AI-generated explanation.

In [5]:
df, movie_embeddings = load_data()

test_cases = [
    'I want a heartwarming family animated movie',
    'dark crime thriller with corrupt police',
    'romantic drama about long-distance love',
]

for user_input in test_cases:
    print(f'\n{"="*60}')
    print(f'USER: "{user_input}"')
    print('='*60)
    results, scores = recommend(user_input, top_k=3, df=df, movie_embeddings=movie_embeddings)
    for i, (_, row) in enumerate(results.iterrows()):
        exp = generate_explanation(user_input, row['title'], row['genres'])
        print(f'\n  {i+1}. {row["title"]} ({row["genres"]})')
        print(f'     Match: {scores[i]:.4f}')
        print(f'     Why:   {exp}')

Loading cached sentence-transformer embeddings...

USER: "I want a heartwarming family animated movie"
Loading sentence transformer: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


  1. Jonah: A VeggieTales Movie (Action Adventure Animation Comedy Family Fantasy Romance)
     Match: 0.6049
     Why:   Jonah: A VeggieTales is a family animated movie.

  2. Doug's 1st Movie (Animation Family Comedy)
     Match: 0.5917
     Why:   Doug's 1st Movie is a family animated movie.

  3. Dwegons (Animation)
     Match: 0.5803
     Why:   Dwegons is a family animated movie.

USER: "dark crime thriller with corrupt police"

  1. I Am Wrath (Action Crime Drama Thriller)
     Match: 0.6940
     Why:   I Am Wrath is a dark crime thriller with corrupt police.

  2. L.A. Confidential (Crime Drama Mystery Thriller)
     Match: 0.6763
     Why:   L.A. Confidential is a dark crime thriller with corrupt police.

  3. The French Connection (Action Crime Thriller)
     Match: 0.6687
     Why:   The French Connection is a dark crime thriller with corrupt police.

USER: "romantic drama about long-distance love"

  1. Brooklyn (Drama Romance)
     Match: 0.5950
     Why:   Brooklyn is a 

## 4. Prompt Variation — Effect on Output Quality

We test three prompt styles to demonstrate that chain-of-thought prompting produces better output than a naive prompt.

In [7]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-base')
model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-base')
model.eval()

def run_prompt(prompt_text):
    inputs = tokenizer(prompt_text, return_tensors='pt', max_length=512, truncation=True)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100, num_beams=4, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

movie = 'Toy Story'
genres = 'Animation|Adventure|Comedy'
mood = 'fun animated movie for children'

prompts = {
    'Naive': f'Why is {movie} a good movie for someone who wants {mood}?',
    'Instructed': f'You are a movie recommender. Explain in 2 sentences why {movie} ({genres}) suits someone who wants: {mood}.',
    'Chain-of-Thought (our method)': _build_prompt(mood, movie, genres),
}

for style, prompt in prompts.items():
    output = run_prompt(prompt)
    print(f'[{style}]')
    print(f'  {output}')
    print()

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


[Naive]
  Toy Story is a good animated movie for someone who wants fun animated movie for children

[Instructed]
  Toy Story (Animation|Adventure|Comedy) is a fun animated movie for children

[Chain-of-Thought (our method)]
  Toy Story is a fun animated movie for children.



## 5. BLEU Score Evaluation

We compute BLEU scores to quantitatively measure how closely the generated explanations match human-written reference descriptions. A higher BLEU score indicates better lexical overlap with the reference.

In [8]:
# Human-written reference explanations (ground truth)
references = [
    ('fun animated movie for children', 'Toy Story', 'Animation|Adventure|Comedy',
     'Toy Story is a beloved animated film perfect for children, blending humour and heartfelt adventure.'),
    ('dark thriller with plot twists', 'The Dark Knight', 'Action|Crime|Drama',
     'The Dark Knight is a gripping crime thriller with complex characters and unexpected plot twists.'),
    ('romantic movie about love and loss', 'Titanic', 'Drama|Romance',
     'Titanic is a sweeping romantic drama exploring love, sacrifice, and loss against a historical backdrop.'),
]

results_data = []
for mood, title, genres, reference in references:
    generated = generate_explanation(mood, title, genres)
    score = bleu_score(reference, generated)
    results_data.append({'Movie': title, 'BLEU Score': round(score, 4),
                         'Generated': generated[:100] + '...'})
    
bleu_df = pd.DataFrame(results_data)
print(bleu_df.to_string(index=False))
print(f'\nMean BLEU Score: {bleu_df["BLEU Score"].mean():.4f}')

          Movie  BLEU Score                                              Generated
      Toy Story      0.1696     Toy Story is a fun animated movie for children....
The Dark Knight      0.2833 The Dark Knight is a dark thriller with plot twists...
        Titanic      0.0692    Titanic is a romantic movie about love and loss....

Mean BLEU Score: 0.1740


## 6. Explanation Consistency — Same Movie, Different Moods

A good explainer should adapt its explanation based on the user's mood, not just the movie itself.

In [9]:
moods = [
    'something funny and lighthearted',
    'a movie about friendship and loyalty',
    'an adventure for young children',
    'feel-good nostalgic film',
]
print('Movie: Toy Story — explanation adapts to different user moods\n')
for mood in moods:
    exp = generate_explanation(mood, 'Toy Story', 'Animation|Adventure|Comedy')
    print(f'Mood : {mood}')
    print(f'Exp  : {exp}')
    print()

Movie: Toy Story — explanation adapts to different user moods

Mood : something funny and lighthearted
Exp  : Toy Story is funny and lighthearted.

Mood : a movie about friendship and loyalty
Exp  : Toy Story is about friendship and loyalty.

Mood : an adventure for young children
Exp  : Toy Story is an adventure for young children.

Mood : feel-good nostalgic film
Exp  : Toy Story is a feel-good nostalgic film.



## Summary

| Component | Detail |
|-----------|--------|
| Model | `google/flan-t5-base` (~250M parameters) |
| Architecture | T5 encoder-decoder (seq2seq) |
| Prompting technique | Chain-of-thought + in-context instruction |
| Output | 2-sentence natural-language explanation |
| Decoding | Beam search (num_beams=4) for coherent output |

**Key findings:**
- Chain-of-thought prompting produces more coherent explanations than naive prompts
- The model adapts explanations based on user mood, not just movie genre
- BLEU score provides a quantitative baseline for explanation quality evaluation